# 38 · Comprobar variables clínicas complementarias y aumento de datos

**Revisión 1.0. Selecciona CPU** y ejecuta las celdas en orden en una sesión nueva. Se comprobarán **KL inicial, cirugía previa y WOMAC total** de las mismas rodillas de desarrollo y la aplicación del aumento leve de intensidad previsto en el plan.

La revisión conserva los tres escenarios clínicos: **edad, sexo e IMC**; esos tres datos más **KL inicial**; y esos tres datos más **cirugía previa y WOMAC**. La imputación y el escalado se calcularán únicamente con los participantes de entrenamiento de cada uno de los **cinco pliegues**. Preparar estas transformaciones no ajusta un modelo predictivo.

Necesitarás las mismas dos carpetas de Drive: la de **registros privados** y la de **desarrollo que contiene `transport.json` y la referencia de la libreta 32**. Las entradas basales adicionales ya están preparadas en esa carpeta de desarrollo; la libreta las localiza y comprueba automáticamente.

Se verifican las imágenes ya preparadas y unas vistas comparativas del aumento. **No se entrena ningún modelo, no se descargan pesos y no se abre la prueba reservada.** Los originales y las particiones se conservan. La comprobación usa CPU; deja GPU y TPU desactivadas.

In [ ]:
from pathlib import Path, PurePosixPath
import json, os, re, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile

CODE_REVISION = '097d1ae9a86988db76ddee672ed3a9984768c5ec'
if not re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION) or CODE_REVISION == '0' * 40:
    raise RuntimeError('Esta copia no tiene una revisión publicada. Usa la libreta final entregada.')
if os.environ.get('COLAB_GPU', '0') == '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona CPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Esta libreta debe ejecutarse personalmente en Google Colab; no es un ensayo local.')
if 'knee' in sys.modules or any(name.startswith('knee.') for name in sys.modules):
    raise RuntimeError('Reinicia el entorno para cargar la revisión fijada de esta libreta.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_methodology38_'))
REPO_DIR = SESSION_ROOT / 'repository'
archive_path = SESSION_ROOT / 'code.zip'
url = 'https://codeload.github.com/AlonsoYess/Knee/zip/' + CODE_REVISION
with urllib.request.urlopen(url, timeout=60) as response, archive_path.open('xb') as target:
    if response.geturl() != url:
        raise RuntimeError('La descarga del código cambió de ubicación.')
    total = 0
    while True:
        chunk = response.read(1024**2)
        if not chunk:
            break
        total += len(chunk)
        if total > 128 * 1024**2:
            raise RuntimeError('El archivo del repositorio supera el límite autorizado.')
        target.write(chunk)
REPO_DIR.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if not members or len(members) > 20000 or sum(i.file_size for i in members) > 512 * 1024**2:
        raise RuntimeError('El contenido del repositorio supera los límites de extracción.')
    seen = set()
    for item in members:
        name = item.filename.rstrip('/')
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or '\0' in name or any(p in ('', '.', '..') for p in parts) or parts[0] != 'Knee-' + CODE_REVISION or name in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Se encontró una ruta no permitida en el repositorio.')
        seen.add(name)
        destination = REPO_DIR.joinpath(*parts[1:])
        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            if len(parts) < 2:
                raise RuntimeError('El archivo del repositorio no tiene una ruta relativa válida.')
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024**2)
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee import methodology_inputs_colab as orchestration

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode:
        detail = re.sub(r'https?://\S+', '<url>', (result.stdout + result.stderr)[-12000:])
        detail = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', detail)
        print(detail)
        raise RuntimeError('La comprobación se detuvo. No ejecutes entrenamiento.')
    return result.stdout

print('Revisión exacta del código preparada. No se abrieron datos de participantes.')


## 1. Preparar el intérprete CPU aislado

Se instalan las mismas versiones fijadas del entorno de comprobación en un intérprete temporal separado del intérprete principal de Colab. Esta preparación puede tardar varios minutos.

In [ ]:
print('Preparando el intérprete CPU aislado para la comprobación, sin entrenamiento…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/check_methodology_inputs_colab.py'), '--operation', 'setup', '--repository-root', str(REPO_DIR), '--destination', str(SESSION_ROOT / 'cpu_runtime')]))
NEURAL_PYTHON = Path(runtime['python_path'])
RUN_ENV = orchestration.runtime_environment(REPO_DIR)
print('Intérprete CPU preparado. No se cargaron pesos ni se entrenaron modelos.')


## 2. Verificar el código con datos sintéticos

Se comprueban la correspondencia de las variables, la separación por participante, las transformaciones por pliegue y las restricciones del aumento de datos. Cualquier fallo o prueba omitida detiene la libreta.

In [ ]:
CPU_TESTS = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_methodology_inputs_colab.py'), '--operation', 'tests', '--repository-root', str(REPO_DIR)], cwd=REPO_DIR, env=RUN_ENV))
print(f"Pruebas sintéticas aprobadas: {CPU_TESTS['tests_run']}; omitidas: {CPU_TESTS['skipped']}.")


## 3. Localizar las mismas carpetas de Drive y verificar las entradas

Pega la carpeta habitual de registros privados y la carpeta de desarrollo que contiene `transport.json`. La libreta reconstruye únicamente el paquete de desarrollo, conserva la referencia de la libreta 32 y comprueba los archivos basales adicionales antes de utilizarlos. No es necesario volver a ejecutar las libretas anteriores.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_ROOT = orchestration.checked_path(input('Misma carpeta existente de KneeOAI con los registros privados: ').strip())
REMOTE_PACKAGE_DIR = orchestration.checked_path(input('Misma carpeta privada de desarrollo, que contiene transport.json: ').strip())
for selected in (PERSISTENT_ROOT, REMOTE_PACKAGE_DIR):
    if not selected.is_dir() or not selected.is_relative_to(Path('/content/drive')):
        raise RuntimeError('Selecciona carpetas existentes dentro de tu Drive montado.')
LOCAL_PACKAGE_DIR = SESSION_ROOT / 'development'
REFERENCE_DIR = SESSION_ROOT / 'reference'
REFERENCE_DIR.mkdir(exist_ok=False)
LOCAL_REFERENCE = REFERENCE_DIR / 'model_inputs_loading_reference_v1.json'
transport = orchestration.stage_development(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR, LOCAL_REFERENCE)
LOCAL_BASELINE_DIR = SESSION_ROOT / 'baseline'
baseline = orchestration.stage_baseline(REMOTE_PACKAGE_DIR, LOCAL_BASELINE_DIR)
LOCAL_PROJECTION = Path(baseline['projection_path'])
LOCAL_DESCRIPTOR = Path(baseline['descriptor_path'])
print('Paquete de desarrollo, referencia y variables basales adicionales verificados.')


## 4. Comprobar las variables clínicas y el aumento de intensidad

Se verifican las entradas de los tres escenarios en cada pliegue. Los valores ausentes se tratan con estadísticas calculadas solo dentro de su entrenamiento. Se revisa que el aumento cambie únicamente la intensidad de copias de entrenamiento, conserve geometría, lateralidad y relleno, y sea reproducible. Validación conserva su transformación determinista.

Esta comprobación no calcula el rendimiento predictivo ni demuestra que el aumento vaya a mejorar un modelo. No inicia ningún entrenamiento.

In [ ]:
import uuid
LOCAL_OUTPUT = SESSION_ROOT / ('methodology_check_' + uuid.uuid4().hex)
print('Comprobando las variables basales de desarrollo, los cinco pliegues y el aumento de intensidad. Solo CPU; sin entrenamiento…')
verification_log = checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_methodology_inputs_colab.py'), '--operation', 'verify', '--repository-root', str(REPO_DIR), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-path', str(LOCAL_REFERENCE), '--projection-path', str(LOCAL_PROJECTION), '--descriptor-path', str(LOCAL_DESCRIPTOR), '--output-dir', str(LOCAL_OUTPUT), '--execution-environment', 'google_colab_cpu'], cwd=REPO_DIR, env=RUN_ENV)
result = json.loads((LOCAL_OUTPUT / orchestration.CORE_FILENAME).read_text(encoding='utf-8'))
orchestration.validate_result(result, 'google_colab_cpu')
print('Comprobación terminada. Se prepararon y verificaron las entradas; no se entrenaron modelos.')


## 5. Revisar las vistas del aumento

Cada vista muestra una imagen de desarrollo antes y después de aplicar el aumento de intensidad. Sirven para revisar que el cambio es leve y conserva el encuadre; no muestran resultados predictivos. Estas vistas y los archivos por rodilla permanecen en el almacenamiento privado.

In [ ]:
from IPython.display import display, Image
for relative in orchestration.PREVIEW_FILES:
    preview = orchestration.checked_path(LOCAL_OUTPUT / relative)
    if not preview.is_file() or not preview.is_relative_to(LOCAL_OUTPUT):
        raise RuntimeError('Falta una vista verificada del aumento de intensidad.')
    display(Image(filename=str(preview)))


## 6. Guardar y verificar el resultado en la misma carpeta de Drive

Se guarda la comprobación completa en una subcarpeta privada nueva, junto a los registros anteriores. El guardado incluye comprobación de integridad y lectura de vuelta. No se reemplazan resultados previos ni se concede autorización para entrenar.

In [ ]:
TEST_RECEIPT = SESSION_ROOT / 'synthetic_test_receipt.json'
RUNTIME_RECEIPT = SESSION_ROOT / 'cpu_setup_receipt.json'
orchestration._save(TEST_RECEIPT, CPU_TESTS)
orchestration._save(RUNTIME_RECEIPT, runtime)
FINAL_CHECK = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_methodology_inputs_colab.py'), '--operation', 'save', '--repository-root', str(REPO_DIR), '--output-dir', str(LOCAL_OUTPUT), '--records-root', str(PERSISTENT_ROOT), '--code-revision', CODE_REVISION, '--test-receipt', str(TEST_RECEIPT), '--runtime-receipt', str(RUNTIME_RECEIPT)], cwd=REPO_DIR, env=RUN_ENV))
orchestration.validate_summary(FINAL_CHECK)
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
print('Comprobación guardada y verificada. No se entrenó ni se abrió la prueba reservada.')


Comparte únicamente el **JSON final**. No necesitas enviar las imágenes ni los archivos privados. Las vistas quedan disponibles para tu revisión; el resultado de esta libreta acredita preparación y comprobaciones de entradas, no entrenamiento ni rendimiento predictivo.